# Create Dataframes from the below Data 

## usecase -1

The data team has provided two CSV files — customer_data.csv and sales_data.csv. Load them into Spark DataFrames, infer schemas automatically, and print the schema along with the first 5 rows of each.

In [0]:
# create both Customer and transaction drataframes from the given source
cust_schema="customer_id string,gender string,age int,payment_method string"
sales_schema="invoice_no string,customer_id string,category string,quantity int,price decimal(10,2),invoice_date string,shopping_mall string"
cust_df=spark.read.format("csv").option("header","true").schema(cust_schema).load("/Volumes/izwd37dev/wd37db/rawdatta/Use_case_30062026/customer_data.csv")
sales_df=spark.read.format("csv").option("header","true").schema(sales_schema).load("/Volumes/izwd37dev/wd37db/rawdatta/Use_case_30062026/sales_data.csv")
cust_df.show(5)
sales_df.show(5)

## usecase- 2

The marketing team wants to know the total revenue generated by the 'Clothing' category for each month.

 Revenue = quantity × price.



In [0]:
#total revenue generated by clothing team

sales_df.createOrReplaceTempView("sales_vw")
cust_df.createOrReplaceTempView("cust_vw")
#spark.sql("select sum(price) from sales where category='Clothing'").show()
#total revenue generated by electronics team
spark.sql("select category, sum(price * quantity) as revenue, sum(price) tot_price, sum(quantity) tot_quantity from sales_vw where upper(category)='CLOTHING' group by category").show()
#total revenue generated by electronics team



## usecase 3

Who is the best customer? Find the customer who has spent the most money overall. Include their customer_id and name (from the customer table). Show the top 5.

In [0]:
spark.sql("select c.customer_id, sum(s.price*s.quantity), sum(price),sum(quantity) from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by c.customer_id order by sum(s.price*s.quantity) desc,c.customer_id limit 5 ").show()

## usecase 4

Generate a customer revenue summary.

 For each customer, show their total revenue and classify them as
  'High Value' (≥ 5000), 
  'Mid Value' (≥ 1000), or 
  'Low Value' (< 1000)

In [0]:
spark.sql("select c.customer_id, sum(s.price*s.quantity) total_revenue, case when sum(s.price*s.quantity) >= 5000 then 'High Value' when sum(s.price*s.quantity) >= 1000 then 'Mid Value'  else 'Low Value' end as Revenue_category from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by c.customer_id order by 1 ").show()

## usecase 5

Is there a significant spending difference between male and female customers? Compute average transaction value, total revenue, and transaction count by gender.

In [0]:
spark.sql("select gender,avg(price) average_transaction,sum(price*quantity) total_revenue_at_gender_level, count(1) transaction_count from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by gender ").show()

## usecase-6

The finance team wants to know which payment method is most popular for each product category. Show the count of transactions per payment method per category.

In [0]:
spark.sql("select category,payment_method,count(1) transaction_count from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by category,payment_method order by transaction_count desc").show()

## usecase 7

Data integrity check: Are there any transactions with a customer_id that does not exist in the customer master table? Identify and count orphaned transactions.


In [0]:
spark.sql("select s.customer_id,count(1) from sales_vw s left join cust_vw c on s.customer_id=c.customer_id where c.customer_id is null group by s.customer_id").show()
spark.sql("select s.customer_id,count(1) from sales_vw s left anti join cust_vw c on s.customer_id=c.customer_id group by s.customer_id").show()

## usecase 8

The marketing team wants to target ads by age group. Bucket customers into: Teens (< 20), Young Adults (20–35), Adults (36–50), Seniors (50+). Which segment generates the most revenue?

In [0]:
spark.sql("select age_group,sum(price*quantity) Revenue from (select case when age<20 then 'Teenager' when age between 20 and 35 then 'Young Adults' when age between 36 and 50 then 'Adults' else 'Senior' end as age_group, price,quantity from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id )group by age_group order by 2 desc").show()

## usecase 9

The retention team wants to run a win-back campaign. Identify all customers whose most recent purchase is more than 90 days before the latest invoice date in the dataset.

consider max_date from the table as current_date

In [0]:
from pyspark.sql.functions import *
spark.sql("select s.customer_id,max(date_format(to_date(invoice_date, 'dd-MM-yyyy'), 'yyyy-MM-dd')) latest_invoice_date,current_date() from sales_vw s inner join cust_vw c on s.customer_id=c.customer_id where date_diff(current_date(),date_format(to_date(invoice_date, 'dd-MM-yyyy'), 'yyyy-MM-dd'))>90 group by s.customer_id ").show()    
          
          
    

## usecase 10

Create a clean, analysis-ready master dataset by joining customer and transaction tables. Derive revenue, month, day-of-week, and age group. Handle nulls. Persist as a Parquet table.

In [0]:
result_df=spark.sql("select sum(price*quantity),invoice_date, month(date_format(to_date(invoice_date, 'dd-MM-yyyy'), 'yyyy-MM-dd')) Month, dayofweek(date_format(to_date(invoice_date, 'dd-MM-yyyy'), 'yyyy-MM-dd')) DAY_of_week from sales_vw s inner join cust_vw c on s.customer_id=c.customer_id group by invoice_date,month(date_format(to_date(invoice_date, 'dd-MM-yyyy'), 'yyyy-MM-dd')), dayofweek(date_format(to_date(invoice_date, 'dd-MM-yyyy'), 'yyyy-MM-dd'))")
display(result_df)  
result_df.write.mode("overwrite").parquet("/Volumes/izwd37dev/wd37db/rawdatta/BB2/result_Use_cases/")

## usecase 11

What is the gender distribution across different product categories

In [0]:
spark.sql("select gender,category, count(1) gen_across_cat from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by gender,category order by 2 ,3 desc,1").show()

## usecase 12

What is the total revenue generated in the year 2022

In [0]:
spark.sql("select sum(price)*sum(quantity) Tot_revenue from sales_vw s inner join cust_vw c on s.customer_id=c.customer_id where year(date_format(to_date(s.invoice_date,'dd-MM-yyyy'),'yyyy-MM-dd'))='2022'").show()

spark.sql("select year(date_format(to_date(s.invoice_date,'dd-MM-yyyy'),'yyyy-MM-dd')) yr, invoice_date from sales_vw s inner join cust_vw c on s.customer_id=c.customer_id where year(date_format(to_date(s.invoice_date,'dd-MM-yyyy'),'yyyy-MM-dd'))='2022'").show()


spark.sql("select year(date_format(to_date(s.invoice_date,'dd-MM-yyyy'),'yyyy-MM-dd')) yr, invoice_date from sales_vw s  where year(date_format(to_date(s.invoice_date,'dd-MM-yyyy'),'yyyy-MM-dd'))='2022'").count()